## Simulation Option 1: Basic using only OpenMM

In [ ]:
# This script was generated by OpenMM-Setup on 2026-10-16.

from openmm import * # import OpenMM
# use the following to access the application layer of the Python API
from openmm.app import *
from openmm.unit import *

# Input Files

pdbx = PDBxFile('../1AKI-processed.cif') # load PDB file of protein generated by OpenMM setup
forcefield = ForceField('amber19-all.xml', 'amber19/tip3pfb.xml') # load force fields

# System Configuration

nonbondedMethod = PME
nonbondedCutoff = 1.0*nanometers
constraints = HBonds
rigidWater = True
constraintTolerance = 0.000001
hydrogenMass = 1.5*amu
ewaldErrorTolerance = 0.0005

# Integration Options

dt = 0.004*picoseconds
temperature = 300*kelvin
friction = 1.0/picosecond
pressure = 1.0*atmospheres
barostatInterval = 25

# Simulation Options

steps = 20000
equilibrationSteps = 1000
# StateDataReporter prints some basic info about the state of the sim to a file during the run
dataReporter = StateDataReporter('log.txt', 100, totalSteps=steps,
    step=True, speed=True, progress=True, potentialEnergy=True, temperature=True, density=True, separator='\t')

# Prepare the Simulation

print('Building system...')
topology = pdbx.topology # topology contains information about the identity and connectivity of particles in the system (i.e., chains and residues)
positions = pdbx.positions # contains coordinates of particles
system = forcefield.createSystem(topology, nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
    constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance, hydrogenMass=hydrogenMass)
system.addForce(MonteCarloBarostat(pressure, temperature, barostatInterval)) # control pressure and temp
integrator = LangevinMiddleIntegrator(temperature, friction, dt) # integration algorithm
integrator.setConstraintTolerance(constraintTolerance)
simulation = Simulation(topology, system, integrator) # create Simulation object to manage the simulation
simulation.context.setPositions(positions) # context object contains info about state of simulation, including particle positions and velocities

# Minimize and Equilibrate

print('Performing energy minimization...')
simulation.minimizeEnergy()
print('Equilibrating...')
simulation.context.setVelocitiesToTemperature(temperature)
simulation.step(equilibrationSteps)

# Simulate

print('Simulating...')
simulation.reporters.append(dataReporter)
simulation.currentStep = 0
simulation.step(steps)

Building system...
Performing energy minimization...
Equilibrating...
Simulating...


In [ ]:
# plots for simulation output
import numpy as np
import matplotlib.pyplot as plt

step, pe, temp, rho = np.loadtxt("log.txt", delimiter="\t", usecols=(1, 2, 3, 4)).T
time = step * dt.value_in_unit(picosecond)

figure = plt.figure(figsize=(8, 2.5), layout="constrained")

axes_pe = figure.add_subplot(1, 3, 1)
axes_pe.plot(time, pe)
axes_pe.set_xlabel("Time (ps)")
axes_pe.set_ylabel("Potential energy (kJ/mol)")

axes_temp = figure.add_subplot(1, 3, 2)
axes_temp.plot(time, temp)
axes_temp.set_xlabel("Time (ps)")
axes_temp.set_ylabel("Temperature (K)")

axes_rho = figure.add_subplot(1, 3, 3)
axes_rho.plot(time, rho)
axes_rho.set_xlabel("Time (ps)")
axes_rho.set_ylabel("Density (g/mL)")

plt.show()

## Simulation Option 2: Use external tool for molecule set up

In [ ]:
# This script was generated by OpenMM-Setup on 2025-09-25.

from openmm import *
from openmm.app import *
from openmm.unit import *

# Input Files

inpcrd = AmberInpcrdFile('input.inpcrd')
prmtop = AmberPrmtopFile('input.prmtop', periodicBoxVectors=inpcrd.boxVectors)

# System Configuration

nonbondedMethod = PME
nonbondedCutoff = 1.0*nanometers
constraints = HBonds
rigidWater = True
constraintTolerance = 0.000001
hydrogenMass = 1.5*amu
ewaldErrorTolerance = 0.0005

# Integration Options

dt = 0.004*picoseconds
temperature = 300*kelvin
friction = 1.0/picosecond
pressure = 1.0*atmospheres
barostatInterval = 25

# Simulation Options

steps = 100000
equilibrationSteps = 10000
dcdReporter = DCDReporter('trajectory.dcd', 10000)
dataReporter = StateDataReporter('log.txt', 1000, totalSteps=steps,
    step=True, speed=True, progress=True, potentialEnergy=True, temperature=True, separator='\t')
checkpointReporter = CheckpointReporter('checkpoint.chk', 10000)

# Prepare the Simulation

print('Building system...')
topology = prmtop.topology
positions = inpcrd.positions
system = prmtop.createSystem(nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
    constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance, hydrogenMass=hydrogenMass)
system.addForce(MonteCarloBarostat(pressure, temperature, barostatInterval))
integrator = LangevinMiddleIntegrator(temperature, friction, dt)
integrator.setConstraintTolerance(constraintTolerance)
simulation = Simulation(topology, system, integrator)
simulation.context.setPositions(positions)

# Minimize and Equilibrate

print('Performing energy minimization...')
simulation.minimizeEnergy()
print('Equilibrating...')
simulation.context.setVelocitiesToTemperature(temperature)
simulation.step(equilibrationSteps)

# Simulate

print('Simulating...')
simulation.reporters.append(dcdReporter)
simulation.reporters.append(dataReporter)
simulation.reporters.append(checkpointReporter)
simulation.currentStep = 0
simulation.step(steps)